In [1]:
%load_ext autoreload
%autoreload 2

import dataclasses
from functools import partial

from absl.testing import parameterized
import jax
import jax.numpy as jnp
from jax.sharding import PartitionSpec as P
import numpy as np
from jax.experimental.scheduling_groups import scheduling_group
from jax.experimental.xla_metadata import set_xla_metadata

import moe
from moe.core import run_moe, add_indices
from moe.ra2a_simulator import ragged_all_to_all as ra2a_via_ag
from moe.pipelined import create_moe, create_moe3
from tests.utils import generate_data

# jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
# jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
# jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)

try:
  jax.config.update("jax_num_cpu_devices", 8)
except RuntimeError:
  pass

In [2]:
experts_per_tok = 8
multiple = 1
devices = jax.devices()
axis_name = "x"
mesh = jax.make_mesh((len(devices),), (axis_name,), axis_types=jax.sharding.AxisType.Explicit, devices=devices)

jax.sharding.set_mesh(mesh)

In [3]:
#n, k, g = 1024, 128, 32
m, k, g = 4096 * 8 * 4, 7168, 32
#m, k, g = 4096 * 4, 7168, 32
x = jax.jit(lambda: jax.random.normal(jax.random.key(0), (m, k), dtype="bfloat16"), out_shardings=P("x", None))()
all_idxs = jax.jit(lambda: jax.random.randint(jax.random.key(0), (experts_per_tok * x.shape[0],), minval=0, maxval=g), out_shardings=P(None))()
x = x.reshape((x.shape[0], 8, -1))
n = 2048

In [4]:
def compute_block(y, group_sizes, *args):
  # construct dummy weight where the weight is just the expert index
  shard_idx = jax.lax.axis_index(axis_name)
  iota = jax.lax.broadcasted_iota("int32", (y.shape[0], group_sizes.shape[-1]), 0)
  starts, ends = jnp.cumsum(group_sizes) - group_sizes, jnp.cumsum(group_sizes)
  assert (g // len(devices)) == group_sizes.size
  group_idxs = group_sizes.size * shard_idx + jnp.arange(group_sizes.size)
  weights = jnp.sum(((iota >= starts[None, :]) & (iota < ends[None, :])) * group_idxs[None, :], -1)
  return y * weights[:, None, None]

In [ ]:
opts = dict(ragged_all_to_all=jax.lax.ragged_all_to_all, axis_name=axis_name, experts_num=g, gathers="builtin")
moe_fn = jax.jit(partial(run_moe, compute_block=compute_block, **opts))
y_ref = moe_fn(x, all_idxs)

### opt barrier

In [10]:
def _schedule(fn, *args):
  cond_fn = lambda val_i: val_i[1] < 1
  body_fn = lambda val_i: (fn(*val_i[0]), val_i[1] + 1)
  fake_dep_fn = lambda x: jax.tree.leaves(x)[0].reshape(-1)[0] * 1e-30
  return jax.lax.while_loop(cond_fn, body_fn, (args, fake_dep_fn(args)))[0]


@partial(jax.jit, static_argnames=("splits",))
def custom_moe(x, all_idxs, splits=1):
  opts = dict(axis_name=axis_name, experts_per_tok=experts_per_tok, experts_num=g, gathers="custom_sc")
  #moe_methods = create_moe(compute_block, ragged_all_to_all=jax.lax.ragged_all_to_all, **opts)
  moe_methods = create_moe(compute_block, ragged_all_to_all=partial(moe.sc_kernels.ra2a, multiple=1), **opts)

  @partial(jax.shard_map, in_specs=((P(axis_name, None, None)), P()), out_specs=P(axis_name, None, None), check_vma=False)
  def inner(x, all_idxs):
    axis_size = jax.lax.axis_size(axis_name)
    assert x.shape[0] % splits == 0
    assert all_idxs.shape[0] % (splits * axis_size) == 0

    x_ = x.reshape((splits, x.shape[0] // splits, *x.shape[1:]))
    all_idxs_ = all_idxs.reshape((axis_size, splits, all_idxs.size // (splits * axis_size)))

    meta = moe_methods.compute_meta(all_idxs)
    y1 = moe_methods.load_fn(x, meta)
    y2 = moe_methods.compute_fn(y1, meta)
    y3 = moe_methods.unload_fn(y2, meta)

    all_metas = [moe_methods.compute_meta(all_idxs_[:, i, ...]) for i in range(splits)]

    x_next = x_[0, ...]
    y1s, y2s, y3s = [], [], []
    for i in range(splits + 2):
      with jax.named_scope(f"iteration_{i}"):
        if (i < splits) and i >= 0:
          y1 = moe_methods.load_fn(x_next, all_metas[i])
        else:
          y1 = None
        if (i < splits + 1) and i >= 1:
          y2 = moe_methods.compute_fn(y1s[i - 1], all_metas[i - 1])
        else:
          y2 = None
        if (i < splits + 2) and i >= 2:
          y3 = moe_methods.unload_fn(y2s[i - 2], all_metas[i - 2])
        else:
          y3 = None
        if i < splits - 1:
          x_next = x_[i, ...]
          y1, y2, y3, x_next = jax.lax.optimization_barrier((y1, y2, y3, x_next))
        else:
          y1, y2, y3 = jax.lax.optimization_barrier((y1, y2, y3))
        if y1 is not None:
          y1s.append(y1)
        if y2 is not None:
          y2s.append(y2)
        if y3 is not None:
          y3s.append(y3)
    return jnp.concat(y3s, axis=0)
  return inner(x, all_idxs)

### old style

In [ ]:
def _schedule(fn, *args):
  cond_fn = lambda val_i: val_i[1] < 1
  body_fn = lambda val_i: (fn(*val_i[0]), val_i[1] + 1)
  fake_dep_fn = lambda x: jax.tree.leaves(x)[0].reshape(-1)[0] * 1e-30
  return jax.lax.while_loop(cond_fn, body_fn, (args, fake_dep_fn(args)))[0]


@partial(jax.jit, static_argnames=("splits",))
def custom_moe_old(x, all_idxs, splits=1):
  opts = dict(axis_name=axis_name, experts_per_tok=experts_per_tok, experts_num=g, gathers="custom_sc")
  #moe_methods = create_moe(compute_block, ragged_all_to_all=jax.lax.ragged_all_to_all, **opts)
  moe_methods = create_moe(compute_block, ragged_all_to_all=partial(moe.sc_kernels.ra2a, multiple=1), **opts)

  @partial(jax.shard_map, in_specs=((P(axis_name, None, None)), P()), out_specs=P(axis_name, None, None), check_vma=False)
  def inner(x, all_idxs):
    axis_size = jax.lax.axis_size(axis_name)
    assert x.shape[0] % splits == 0
    assert all_idxs.shape[0] % (splits * axis_size) == 0

    x_ = x.reshape((splits, x.shape[0] // splits, *x.shape[1:]))
    all_idxs_ = all_idxs.reshape((axis_size, splits, all_idxs.size // (splits * axis_size)))

    meta = moe_methods.compute_meta(all_idxs)

    outputs = []

    def body_fn(x, all_idxs):
      meta = moe_methods.compute_meta(all_idxs)
      y = moe_methods.load_fn(x, meta)
      y = moe_methods.compute_fn(y, meta)
      y = moe_methods.unload_fn(y, meta)
      return y, all_idxs

    for i in range(splits):
      #meta = moe_methods.compute_meta(all_idxs_[:, i, ...])
      ##with set_xla_metadata(_scheduling_group_id=0 + i):
      #y = moe_methods.load_fn(x_[i, ...], meta)
      ##with set_xla_metadata(_scheduling_group_id=1 + i):
      #y = moe_methods.compute_fn(y, meta)
      ##with set_xla_metadata(_scheduling_group_id=2 + i):
      #y = moe_methods.unload_fn(y, meta)

      args = (x_[i, ...], all_idxs_[:, i, ...])
      y, _ = _schedule(body_fn, *args)
      #y, _ = body_fn(*args)
      #y, _ = jax.lax.cond(jnp.array(1) == 1, body_fn, lambda *args: args, *args)
      outputs.append(y)

    return jnp.concat(outputs, axis=0)
  return inner(x, all_idxs)

### new style

In [ ]:
def _schedule(fn, *args):
  cond_fn = lambda val_i: val_i[1] < 1
  body_fn = lambda val_i: (fn(*val_i[0]), val_i[1] + 1)
  return jax.lax.while_loop(cond_fn, body_fn, (args, 0))[0]


opts = dict(axis_name=axis_name, experts_per_tok=experts_per_tok, experts_num=g, gathers="custom_sc")
#moe_methods = create_moe(compute_block, ragged_all_to_all=jax.lax.ragged_all_to_all, **opts)
moe_methods = create_moe(compute_block, ragged_all_to_all=partial(moe.sc_kernels.ra2a, multiple=1), **opts)


@jax.jit
#@scheduling_group(name="0")
@partial(jax.shard_map, in_specs=P(), out_specs=P(axis_name), check_vma=False)
def zeroth(all_idxs):
  return moe_methods.compute_meta(all_idxs)


@jax.jit
#@scheduling_group(name="1")
@partial(jax.shard_map, in_specs=((P(axis_name, None, None)), P(axis_name)), out_specs=P(axis_name, None, None), check_vma=False)
def first(x, meta):
  return moe_methods.load_fn(x, meta)


@jax.jit
#@scheduling_group(name="2")
@partial(jax.shard_map, in_specs=((P(axis_name, None, None)), P(axis_name)), out_specs=P(axis_name, None, None), check_vma=False)
def second(y, meta):
  return moe_methods.compute_fn(y, meta)


@jax.jit
#@scheduling_group(name="3")
@partial(jax.shard_map, in_specs=((P(axis_name, None, None)), P(axis_name)), out_specs=P(axis_name, None, None), check_vma=False)
def third(y, meta):
  return moe_methods.unload_fn(y, meta)


#@partial(jax.jit, static_argnames=("splits",))
def custom_moe_nonjit(x, all_idxs, splits=1):
  meta = zeroth(all_idxs)
  y = first(x, meta)
  y = second(y, meta)
  y = third(y, meta)
  return y

### testing the custom moe

In [33]:
custom_moe_ = custom_moe.lower(x, all_idxs).compile({"xla_tpu_scheduling_annotation_deannotate_unsupported_groups": False})
custom_moe2_ = jax.jit(partial(custom_moe, splits=8)).lower(x, all_idxs).compile({"xla_tpu_scheduling_annotation_deannotate_unsupported_groups": False})

In [34]:
y = custom_moe_(x, all_idxs)
#y = custom_moe(x, all_idxs)

In [35]:
jnp.mean(jnp.abs(y_ref - y) == 0)

Array(1., dtype=float32)

In [36]:
y = jax.block_until_ready(custom_moe_(x, all_idxs))
y = jax.block_until_ready(custom_moe2_(x, all_idxs))
#y_ref = jax.block_until_ready(moe_fn(x, all_idxs))
with moe.utils.profile():
  for _ in range(3):
    jax.block_until_ready(custom_moe_(x, all_idxs))
  for _ in range(3):
    jax.block_until_ready(custom_moe2_(x, all_idxs))
  #for _ in range(4):
  #  jax.block_until_ready(moe_fn(x, all_idxs))

http://localhost:52436/data/plugin/profile/trace_viewer@;run=2025_11_29_04_48_58;tag=trace_viewer@


## actual compute block

In [5]:
# w1 = jnp.ones((g, k, n), dtype="bfloat16", out_sharding=P(axis_name, None, None)) / (k + n)
# w2 = jnp.ones((g, k, n), dtype="bfloat16", out_sharding=P(axis_name, None, None)) / (k + n)
# w3 = jnp.ones((g, n, k), dtype="bfloat16", out_sharding=P(axis_name, None, None)) / (n + k)
keys = iter(jax.random.split(jax.random.key(7), 1024))
w1 = jax.random.normal(next(keys), (g, k, n), dtype="bfloat16", out_sharding=P(axis_name, None, None)) / (k + n) ** 0.5
w2 = jax.random.normal(next(keys), (g, k, n), dtype="bfloat16", out_sharding=P(axis_name, None, None)) / (k + n) ** 0.5
w3 = jax.random.normal(next(keys), (g, n, k), dtype="bfloat16", out_sharding=P(axis_name, None, None)) / (n + k) ** 0.5

In [6]:
def compute_block(y, group_sizes, w1, w2, w3):
  # jax.debug.print("group_sizes = {}, pct of total = {}%", group_sizes, jnp.sum(group_sizes) / y.shape[0] * 1e2)
  y_shape = y.shape
  y = y.reshape((y.shape[0], -1))
  with set_xla_metadata(ragged_dot_tiling="1024,1024,1024"):
    y1 = jax.lax.ragged_dot(y, w1, group_sizes)
    y2 = jax.lax.ragged_dot(y, w2, group_sizes)
    y3 = y1 * jax.nn.gelu(y2, approximate=True)
    y4 = jax.lax.ragged_dot(y3, w3, group_sizes)
  return y4.reshape(y_shape)

### old attempt

In [9]:
@partial(jax.jit, static_argnames=("splits",))
def custom_moe(x, all_idxs, *extra_args, splits=1):
  opts = dict(axis_name=axis_name, experts_per_tok=experts_per_tok, experts_num=g, gathers="custom_sc")
  moe_methods = create_moe(compute_block, ragged_all_to_all=partial(moe.sc_kernels.ra2a, multiple=1), **opts)

  extra_specs = jax.tree.map(lambda x: jax.typeof(x).sharding.spec, extra_args)

  @partial(jax.shard_map, in_specs=((P(axis_name, None, None)), P(), *extra_specs), out_specs=P(axis_name, None, None), check_vma=False)
  def inner(x, all_idxs, *extra_args):
    axis_size = jax.lax.axis_size(axis_name)
    assert x.shape[0] % splits == 0
    assert all_idxs.shape[0] % (splits * axis_size) == 0

    x_ = x.reshape((splits, x.shape[0] // splits, *x.shape[1:]))
    all_idxs_ = all_idxs.reshape((axis_size, splits, all_idxs.size // (splits * axis_size)))

    all_metas = [moe_methods.compute_meta(all_idxs_[:, i, ...]) for i in range(splits)]
    x_next = x_[0, ...]
    y1s, y2s, y3s = [], [], []
    for i in range(splits + 2):
      with jax.named_scope(f"iteration_{i}"):
        y1 = moe_methods.load_fn(x_next, all_metas[i]) if ((i < splits) and i >= 0) else None
        y2 = moe_methods.compute_fn(y1s[i - 1], all_metas[i - 1], *extra_args) if ((i < splits + 1) and i >= 1) else None
        y3 = moe_methods.unload_fn(y2s[i - 2], all_metas[i - 2]) if ((i < splits + 2) and i >= 2) else None
        if i < splits - 1:
          x_next = x_[i, ...]
          #y1, y2, y3, x_next = jax.lax.optimization_barrier((y1, y2, y3, x_next))
        else:
          #y1, y2, y3 = jax.lax.optimization_barrier((y1, y2, y3))
          pass
        y1s.append(y1) if y1 is not None else None
        y2s.append(y2) if y2 is not None else None
        y3s.append(y3) if y3 is not None else None
    return jnp.concat(y3s, axis=0)
  return inner(x, all_idxs, *extra_args)

In [ ]:
#@partial(jax.jit, in_shardings=(P(axis_name), P(axis_name)))
#def custom_moe(x, meta):
@partial(jax.jit, in_shardings=(P(axis_name), P()))
def custom_moe(x, all_idxs):

  opts = dict(axis_name=axis_name, experts_per_tok=experts_per_tok, experts_num=g, gathers="custom")
  moe_methods = create_moe(compute_block, ragged_all_to_all=jax.lax.ragged_all_to_all, **opts)

  @partial(jax.shard_map, out_specs=P(axis_name), check_vma=False)
  def inner(x, all_idxs):
    meta = moe_methods.compute_meta(all_idxs)
    #meta = meta.preamble
    #return meta

    #start_fn, wait_fn = moe.ra2a.make_ra2a_3d(axis_name)
    #buffer = jnp.zeros((x.shape[0] * 4, *x.shape[1:]), x.dtype)
    #out, sems = start_fn(x, buffer, *dataclasses.astuple(meta))
    #out = wait_fn(x, out, *dataclasses.astuple(meta), sems)
    #return out
    start_fn, wait_fn = moe_methods.load_fn()
    out = start_fn(x, meta)
    y = wait_fn(out, meta)
    return y

  #return inner(x, meta)
  return inner(x, all_idxs)

In [11]:
x, meta = generate_data(8 * 8 * 4096, 7168, len(jax.devices()), axis_name=axis_name)
x = x.reshape((x.shape[0], 8, -1))

In [12]:
#m, k, g = 4096 * 8 * 4, 7168, 32
m, k, g = 1024, 7168, 32
experts_per_tok = 8
_, meta = generate_data(m, k, len(jax.devices()), axis_name=axis_name)
x = jax.jit(lambda: jax.random.normal(jax.random.key(0), (m, k), dtype="bfloat16"), out_shardings=P("x", None))()
all_idxs = jax.jit(lambda: jax.random.randint(jax.random.key(0), (experts_per_tok * x.shape[0],), minval=0, maxval=g), out_shardings=P(None))()
x = x.reshape((x.shape[0], 8, -1))

In [13]:
meta.input_offsets.reshape((4, -1))

Array([[  0,  68, 129, 182],
       [  0,  64, 131, 200],
       [  0,  76, 137, 198],
       [  0,  54, 122, 185]], dtype=int32)

In [14]:
meta.recv_sizes.reshape((4, -1))

Array([[68, 64, 76, 54],
       [61, 67, 61, 68],
       [53, 69, 61, 63],
       [74, 56, 58, 71]], dtype=int32)

In [18]:
preamble = custom_moe(x, all_idxs)

ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?


In [19]:
#preamble = custom_moe(x, all_idxs)
y = custom_moe(x, all_idxs)
#y = custom_moe(x, meta)
y = jax.block_until_ready(y)
print(jnp.sum(y))

nan


### tune gmm

In [65]:
gs = jnp.round(m * jax.nn.softmax(jax.random.normal(jax.random.key(0), (g,), "bfloat16"), axis=-1)).astype(jnp.int32)
while jnp.sum(gs) != m:
  idx = jnp.argmax(gs)
  gs = gs.at[idx].set(gs[idx] + m - jnp.sum(gs))
gs = jax.device_put(gs, P(axis_name))

x_, meta = generate_data(4096 * 8 * 8 * 4, 7168, 4, axis_name=axis_name)
x_ = x_.reshape((x_.shape[0], 8, -1))
x_up, x_down = x_, x_[..., :2048 // 8]

In [ ]:
import tune_jax
tune_jax.logger.setLevel("INFO")


def gmm(lhs, rhs, group_sizes, tile_m, tile_k, tile_n):
  @partial(jax.shard_map, out_specs=(P(axis_name, None)))
  def _gmm(lhs, rhs, group_sizes):
    with set_xla_metadata(ragged_dot_tiling=f"{tile_m},{tile_k},{tile_n}"):
      return jax.lax.ragged_dot(lhs, rhs, group_sizes)
  return _gmm(lhs, rhs, group_sizes)


hyperparams = dict(
  tile_m=[64, 128, 256, 512],
  tile_k=[7168 // (2 ** i) for i in range(5)] + [2048 // (2 ** i) for i in range(5)],
  tile_n=[256, 512, 1024, 2048] + [7168 // (2 ** i) for i in range(5)],
)

fn = tune_jax.tune(gmm, hyperparams=hyperparams)
fn(x_up.reshape((x_up.shape[0], -1)), w1, gs)
print(tune_jax.tabulate(fn))
fn(x_down.reshape((x_down.shape[0], -1)), w3, gs)
print(tune_jax.tabulate(fn))

### analyze overlap

In [ ]:

@jax.jit
def overlap(lhs, rhs, gs, x, meta):
  @partial(jax.shard_map, out_specs=(P(axis_name, None), P(axis_name, None, None)), check_vma=False)
  def _gmm(lhs, rhs, gs, x, meta):
    #with set_xla_metadata(_scheduling_group_id=17):

    @scheduling_group("group1")
    def fn(lhs, rhs, gs, x, meta):
      idx = jnp.argsort(jnp.arange(x.shape[0]))
      #lhs = lhs[idx, ...]
      x = moe.sc_kernels.sc_gather(x, idx)
      with set_xla_metadata(ragged_dot_tiling=f"{256},{7168},{512}"):
        y = jax.lax.ragged_dot(lhs, rhs, gs)
      output = jax.lax.empty((x.shape[0] + 4 * 1024, *x.shape[1:]), x.dtype)
      # y2 = moe.sc_kernels.ra2a(x, output, *dataclasses.astuple(meta), axis_name=axis_name, multiple=1)
      y2 = jax.lax.ragged_all_to_all(x, output, *dataclasses.astuple(meta), axis_name=axis_name)
      return y, y2

    y, y2 = fn(lhs, rhs, gs, x, meta)

    #y2 = jax.lax.ragged_all_to_all(x, output, *dataclasses.astuple(meta), axis_name=axis_name)

    #with set_xla_metadata(_scheduling_group_id=18):
    @scheduling_group("group1")
    def fn2(lhs, rhs, gs, x, meta):
      with set_xla_metadata(ragged_dot_tiling=f"{256},{7168},{512}"):
        y3 = jax.lax.ragged_dot(lhs, 2 * rhs, gs)
      output = jax.lax.empty((x.shape[0] + 4 * 1024, *x.shape[1:]), x.dtype)
      #y4 = moe.sc_kernels.ra2a(x, output, *dataclasses.astuple(meta), axis_name=axis_name, multiple=1)
      y4 = jax.lax.ragged_all_to_all(x, output, *dataclasses.astuple(meta), axis_name=axis_name)
      return y3, y4

    y3, y4 = fn2(lhs, rhs, gs, x, meta)

    return y + y3, y2 + y4
  return _gmm(lhs, rhs, gs, x, meta)

In [ ]:
_ = overlap(x.reshape((x.shape[0], -1)), w1, gs, x_, meta)

In [58]:
c = overlap.lower(x.reshape((x.shape[0], -1)), w1, gs, x_, meta).compile()

In [59]:
[k for k in dir(c) if not k.startswith("_")]

['args_info',
 'as_text',
 'call',
 'cost_analysis',
 'donate_argnums',
 'in_avals',
 'in_tree',
 'input_formats',
 'input_shardings',
 'memory_analysis',
 'out_info',
 'out_tree',
 'output_formats',
 'output_shardings',
 'runtime_executable']

In [49]:
[k for k in dir(c.memory_analysis()) if not k.startswith("_")]

['alias_size_in_bytes',
 'argument_size_in_bytes',
 'generated_code_size_in_bytes',
 'host_alias_size_in_bytes',
 'host_argument_size_in_bytes',
 'host_generated_code_size_in_bytes',
 'host_output_size_in_bytes',
 'host_temp_size_in_bytes',
 'output_size_in_bytes',
 'peak_memory_in_bytes',
 'serialized_buffer_assignment_proto',
 'temp_size_in_bytes']

In [24]:
[k for k in dir(c.cost_analysis()) if not k.startswith("_")]

['clear',
 'copy',
 'fromkeys',
 'get',
 'items',
 'keys',
 'pop',
 'popitem',
 'setdefault',
 'update',
 'values']

In [25]:
c.cost_analysis()

{'bytes accessed2{}': 2048.0,
 'bytes accessed': 42756747264.0,
 'bytes accessed3{}': 2048.0,
 'bytes accessed4{}': 7516193280.0,
 'bytes accessed5{}': 8103395328.0,
 'bytes accessedout{}': 11601449984.0,
 'utilization6{}': 1.0,
 'utilization5{}': 3.0,
 'bytes accessed1{}': 3951037952.0,
 'utilization4{}': 3.0,
 'utilization2{}': 3.0,
 'bytes accessed6{}': 234881024.0,
 'optimal_seconds': -3.9733612537384033,
 'bytes accessed0{}': 11467233280.0,
 'utilization0{}': 33.0,
 'flops': 964048191488.0,
 'utilization3{}': 3.0,
 'utilization1{}': 10.0}

In [60]:
jax.block_until_ready(overlap(x.reshape((x.shape[0], -1)), w1, gs, x_, meta))
with moe.utils.profile():
  for _ in range(2):
    jax.block_until_ready(overlap(x.reshape((x.shape[0], -1)), w1, gs, x_, meta))

http://localhost:52437/data/plugin/profile/trace_viewer@;run=2025_11_29_21_04_57;tag=trace_viewer@


In [17]:
y = jax.block_until_ready(custom_moe(x, all_idxs, w1, w2, w3, splits=2))

### profile final

#### older final

In [30]:
def fn(moe_methods, i, splits, y1, y2, meta1, meta2, meta3, x_next, *extra_args):
  if 0 <= i < splits:
    #start_fn, wait_fn = moe_methods.load_fn()
    #fut = start_fn(x_next, meta1)
    #y1, fut = jax.lax.optimization_barrier((y1, fut))
    #y1_next = wait_fn(fut, meta1)
    y1, y1_next = moe_methods.load_fn(y1, x_next, meta1)
  else:
    y1_next = None

  #y1_next = moe_methods.load_fn(x_next, meta1) if 0 <= i < splits else None

  if 2 <= i < splits + 2:
    #start_fn, wait_fn = moe_methods.unload_fn()
    #fut = start_fn(y2, meta3)
    #(y1, fut) = jax.lax.optimization_barrier((y1, fut))
    #y3_next = wait_fn(fut, meta3)
    y1, y3_next = moe_methods.unload_fn(y1, y2, meta3)
  else:
    y3_next = None
  #y3_next = moe_methods.unload_fn(y2, meta3) if 2 <= i < splits + 2 else None

  y2_next = moe_methods.compute_fn(y1, meta2, *extra_args) if 1 <= i < splits + 1 else None

  #y2_next, y1_next = jax.lax.optimization_barrier((y2_next, y1_next))

  #y3_next, y2_next, y1_next = jax.lax.optimization_barrier((y3_next, y2_next, y1_next))

  return y1_next, y2_next, y3_next


@partial(jax.jit, static_argnames=("splits",))
def custom_moe(all_idxs, x, *extra_args, splits=1):
  opts = dict(axis_name=axis_name, experts_per_tok=experts_per_tok, experts_num=g, gathers="custom")
  #moe_methods = create_moe(compute_block, ragged_all_to_all=partial(moe.sc_kernels.ra2a, multiple=1), **opts)
  #opts = dict(axis_name=axis_name, experts_per_tok=experts_per_tok, experts_num=g, gathers="builtin")
  moe_methods = create_moe(compute_block, ragged_all_to_all=jax.lax.ragged_all_to_all, **opts)

  extra_specs = jax.tree.map(lambda x: jax.typeof(x).sharding.spec, extra_args)

  @partial(jax.shard_map, in_specs=((P(axis_name, None, None)), P(), *extra_specs), out_specs=P(axis_name, None, None), check_vma=False)
  def inner(x, all_idxs, *extra_args):
    axis_size = jax.lax.axis_size(axis_name)
    assert x.shape[0] % splits == 0
    assert all_idxs.shape[0] % (splits * axis_size) == 0

    x_ = x.reshape((splits, x.shape[0] // splits, *x.shape[1:]))
    all_idxs_ = all_idxs.reshape((axis_size, splits, all_idxs.size // (splits * axis_size)))

    all_metas = [moe_methods.compute_meta(all_idxs_[:, i, ...]) for i in range(splits)]
    x_next = x_[0, ...]
    y1s, y2s, y3s = [], [], []
    for i in range(splits + 2):
      #with jax.named_scope(f"iteration_{i:02d}"):

      #fn_ = xla_metadata_call(inlineable="false")(scheduling_group(name=f"group_{i:02d}")(partial(fn, moe_methods, i, splits)))
      fn_ = partial(fn, moe_methods, i, splits)

      meta1 = all_metas[i] if 0 <= i < splits else 0
      y1, meta2 = (y1s[i - 1], all_metas[i - 1]) if 1 <= i < splits + 1 else (None, None)
      y2, meta3 = (y2s[i - 2], all_metas[i - 2]) if 2 <= i < splits + 2 else (None, None)
      y1, y2, y3 = fn_(y1, y2, meta1, meta2, meta3, x_next, *extra_args)
      x_next = x_[i + 1, ...] if (i < splits - 1) else None

      y1, y2, y3, x_next = jax.lax.optimization_barrier((y1, y2, y3, x_next))

      print("new change")
      print(f"{int(y1 is not None)} {int(y2 is not None)} {int(y3 is not None)}")

      y1s.append(y1) if y1 is not None else None
      y2s.append(y2) if y2 is not None else None
      y3s.append(y3) if y3 is not None else None
    return jnp.concat(y3s, axis=0)
  return inner(x, all_idxs, *extra_args)

#### split all vjp finale

In [7]:
def overlap_fn(moe_methods, i, splits, y1, y2, meta1, meta2, meta3, x_next, *extra_args):
  fut1, fut3, y1_next, y3_next = None, None, None, None
  if 0 <= i < splits:
    start_fn1, wait_fn1 = moe_methods.load_fn()
    fut1 = start_fn1(x_next, meta1)
    fut1 = tuple(fut1) + dataclasses.astuple(meta1.preamble)

  if 2 <= i < splits + 2:
    start_fn3, wait_fn3 = moe_methods.unload_fn()
    fut3 = start_fn3(y2, meta3)
    fut3 = tuple(fut3) + dataclasses.astuple(meta3.epilogue)

  compute_fn = moe_methods.compute_fn if 1 <= i < splits + 1 else None
  ra2a_split = moe.ra2a.make_split_ra2a(compute_fn)

  (y1_next, y3_next), y2_next = ra2a_split((fut1, fut3), (y1, meta2, *extra_args), axis_name=axis_name)

  if 0 <= i < splits:
    y1_next = wait_fn1(y1_next, meta1)
  if 2 <= i < splits + 2:
    y3_next = wait_fn3(y3_next, meta3)

  return y1_next, y2_next, y3_next


@partial(jax.jit, static_argnames=("splits",))
def custom_moe(all_idxs, x, *extra_args, splits=1):
  opts = dict(axis_name=axis_name, experts_per_tok=experts_per_tok, experts_num=g, gathers="custom")
  moe_methods = create_moe3(compute_block, **opts)

  extra_specs = jax.tree.map(lambda x: jax.typeof(x).sharding.spec, extra_args)

  @partial(jax.shard_map, in_specs=((P(axis_name, None, None)), P(), *extra_specs), out_specs=P(axis_name, None, None), check_vma=False)
  def inner(x, all_idxs, *extra_args):
    axis_size = jax.lax.axis_size(axis_name)
    assert x.shape[0] % splits == 0
    assert all_idxs.shape[0] % (splits * axis_size) == 0

    x_ = x.reshape((splits, x.shape[0] // splits, *x.shape[1:]))
    all_idxs_ = all_idxs.reshape((axis_size, splits, all_idxs.size // (splits * axis_size)))

    all_metas = [moe_methods.compute_meta(all_idxs_[:, i, ...]) for i in range(splits)]
    x_next = x_[0, ...]
    y1s, y2s, y3s = [], [], []
    for i in range(splits + 2):
      overlap_fn_ = partial(overlap_fn, moe_methods, i, splits)

      meta1 = all_metas[i] if 0 <= i < splits else 0
      y1, meta2 = (y1s[i - 1], all_metas[i - 1]) if 1 <= i < splits + 1 else (None, None)
      y2, meta3 = (y2s[i - 2], all_metas[i - 2]) if 2 <= i < splits + 2 else (None, None)
      y1, y2, y3 = overlap_fn_(y1, y2, meta1, meta2, meta3, x_next, *extra_args)
      x_next = x_[i + 1, ...] if (i < splits - 1) else None

      y1, y2, y3, x_next = jax.lax.optimization_barrier((y1, y2, y3, x_next))

      # print("new change")
      # print(f"{int(y1 is not None)} {int(y2 is not None)} {int(y3 is not None)}")

      y1s.append(y1) if y1 is not None else None
      y2s.append(y2) if y2 is not None else None
      y3s.append(y3) if y3 is not None else None
    return jnp.concat(y3s, axis=0)
  return inner(x, all_idxs, *extra_args)

In [ ]:
opts = dict(ragged_all_to_all=jax.lax.ragged_all_to_all, axis_name=axis_name, experts_num=g, gathers="builtin")
moe_fn = jax.jit(partial(run_moe, compute_block=compute_block, **opts))
y_ref = moe_fn(x, all_idxs, w1, w2, w3)

In [9]:
y = jax.block_until_ready(custom_moe(all_idxs, x, w1, w2, w3, splits=1))

ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?


ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?


fn (((ShapedArray(bfloat16[262144,8,896]), ShapedArray(bfloat16[315392,8,896]), ShapedArray(int32[4]), ShapedArray(int32[4]), ShapedArray(int32[4]), ShapedArray(int32[4])), None), (None, None, ShapedArray(bfloat16[8,7168,2048]), ShapedArray(bfloat16[8,7168,2048]), ShapedArray(bfloat16[8,2048,7168])))
([ShapedArray(bfloat16[315392,8,896]), None], None)
fn ((None, None), (ShapedArray(bfloat16[315392,8,896]), MoEMeta(info=MoEInfo(batch_size=32768, experts_per_tok=8, num_experts=32), local_ra2a_sort=ShapedArray(int32[262144]), local_ra2a_isort=ShapedArray(int32[262144]), preamble=RA2AMeta(input_offsets=ShapedArray(int32[4]), send_sizes=ShapedArray(int32[4]), output_offsets=ShapedArray(int32[4]), recv_sizes=ShapedArray(int32[4])), epilogue=RA2AMeta(input_offsets=ShapedArray(int32[4]), send_sizes=ShapedArray(int32[4]), output_offsets=ShapedArray(int32[4]), recv_sizes=ShapedArray(int32[4])), local_permute=PaddedGroupPaddedMetadata(group_idx=ShapedArray(int32[315392]), group_idx_with_padding=S

In [10]:
y2 = jax.block_until_ready(custom_moe(all_idxs, x, w1, w2, w3, splits=4))

ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?

fn (((ShapedArray(bfloat16[65536,8,896]), ShapedArray(bfloat16[78848,8,896]), ShapedArray(int32[4]), ShapedArray(int32[4]), ShapedArray(int32[4]), ShapedArray(int32[4])), None), (None, None, ShapedArray(bfloat16[8,7168,2048]), ShapedArray(bfloat16[8,7168,2048]), ShapedArray(bfloat16[8,2048,7168])))
([ShapedArray(bfloat16[78848,8,896]), None], None)
fn (((ShapedArray(bfloat16[65536,8,896]), ShapedArray(bfloat16[78848,8,896]), ShapedArray(int32[4]), ShapedArray(int32[4]), ShapedArray(int32[4]), ShapedArray(int32[4])), None), (ShapedArray(bfloat16[78848,8,896]), MoEMeta(info=MoEInfo(batch_size=8192, experts_per_tok=8, num_experts=32), local_ra2a_sort=ShapedArray(int32[65536]), local_ra2a_isort=ShapedArray(int32[65536]), preamble=RA2AMeta(input_offsets=ShapedArray(int32[4]), send_sizes=ShapedArray(int32[4]), output_offsets=ShapedArray(int32[4]), recv_sizes=ShapedArray(int32[4])), epilogue=RA2AMeta(input_offsets=ShapedArray(int32[4]), send_sizes=ShapedArray(int32[4]), output_offsets=ShapedA

In [11]:
print(jnp.sum(jnp.abs(y_ref - y)))
print(jnp.sum(jnp.abs(y_ref - y2)))

0
0


In [12]:
y = jax.block_until_ready(custom_moe(all_idxs, x, w1, w2, w3))
#y = jax.block_until_ready(custom_moe(all_idxs, x, w1, w2, w3, splits=2))
y2 = jax.block_until_ready(custom_moe(all_idxs, x, w1, w2, w3, splits=4))
#y = jax.block_until_ready(custom_moe(all_idxs, x, w1, w2, w3, splits=8))
#y_ref = jax.block_until_ready(moe_fn(all_idxs, x))
with moe.utils.profile():
  for _ in range(3):
    jax.block_until_ready(custom_moe(all_idxs, x, w1, w2, w3))
  #for _ in range(3):
  #  jax.block_until_ready(custom_moe(all_idxs, x, w1, w2, w3, splits=2))
  for _ in range(3):
    jax.block_until_ready(custom_moe(all_idxs, x, w1, w2, w3, splits=4))
  #for _ in range(3):
  #  jax.block_until_ready(custom_moe(all_idxs, x, w1, w2, w3, splits=8))
  #for _ in range(4):
  #  jax.block_until_ready(moe_fn(all_idxs, x))

ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?


fn (((ShapedArray(bfloat16[262144,8,896]), ShapedArray(bfloat16[315392,8,896]), ShapedArray(int32[4]), ShapedArray(int32[4]), ShapedArray(int32[4]), ShapedArray(int32[4])), None), (None, None, ShapedArray(bfloat16[8,7168,2048]), ShapedArray(bfloat16[8,7168,2048]), ShapedArray(bfloat16[8,2048,7168])))
([ShapedArray(bfloat16[315392,8,896]), None], None)
fn ((None, None), (ShapedArray(bfloat16[315392,8,896]), MoEMeta(info=MoEInfo(batch_size=32768, experts_per_tok=8, num_experts=32), local_ra2a_sort=ShapedArray(int32[262144]), local_ra2a_isort=ShapedArray(int32[262144]), preamble=RA2AMeta(input_offsets=ShapedArray(int32[4]), send_sizes=ShapedArray(int32[4]), output_offsets=ShapedArray(int32[4]), recv_sizes=ShapedArray(int32[4])), epilogue=RA2AMeta(input_offsets=ShapedArray(int32[4]), send_sizes=ShapedArray(int32[4]), output_offsets=ShapedArray(int32[4]), recv_sizes=ShapedArray(int32[4])), local_permute=PaddedGroupPaddedMetadata(group_idx=ShapedArray(int32[315392]), group_idx_with_padding=S

In [13]:
jnp.sum(jnp.abs(y - y2))

Array(0, dtype=bfloat16)

In [8]:
@partial(jax.jit)
def fwd_and_bwd_ref(all_idxs, x, w1, w2, w3):
  moe_fn = partial(run_moe, axis_name=axis_name, experts_num=g, compute_block=compute_block)
  o, vjp_fn = jax.vjp(lambda x, *args: moe_fn(x, all_idxs, *args), x, w1, w2, w3)
  r = 2 * o
  return o, vjp_fn(r)

In [18]:
#o_ref, do_ref = jax.block_until_ready(fwd_and_bwd_ref(all_idxs, x, w1, w2, w3))
#del o_ref, do_ref
jax.block_until_ready(fwd_and_bwd_ref(all_idxs, x, w1, w2, w3))
pass

In [9]:
@partial(jax.jit, static_argnames=("splits",))
def fwd_and_bwd(all_idxs, x, w1, w2, w3, splits: int = 1):
  o, vjp_fn = jax.vjp(partial(custom_moe, all_idxs, splits=splits), x, w1, w2, w3)
  r = 2 * o
  sum_ = partial(jnp.sum, axis=(-1, -2))
  return jax.tree.map(sum_, (o, vjp_fn(r)))

In [11]:
#o, do = jax.block_until_ready(fwd_and_bwd(all_idxs, x, w1, w2, w3))
jax.block_until_ready(fwd_and_bwd(all_idxs, x, w1, w2, w3))
#jax.block_until_ready(fwd_and_bwd(all_idxs, x, w1, w2, w3, splits=2))
jax.block_until_ready(fwd_and_bwd(all_idxs, x, w1, w2, w3, splits=4))
#jax.block_until_ready(fwd_and_bwd(all_idxs, x, w1, w2, w3, splits=8))
#o2, do2 = jax.block_until_ready(fwd_and_bwd(all_idxs, x, w1, w2, w3, splits=4))
#y_ref = jax.block_until_ready(moe_fn(all_idxs, x))
#del o, do, o2, do2
with moe.utils.profile():
  for _ in range(2):
    jax.block_until_ready(fwd_and_bwd(all_idxs, x, w1, w2, w3))
  #for _ in range(2):
  #  jax.block_until_ready(fwd_and_bwd(all_idxs, x, w1, w2, w3, splits=2))
  for _ in range(2):
    jax.block_until_ready(fwd_and_bwd(all_idxs, x, w1, w2, w3, splits=4))
  for _ in range(2):
    jax.block_until_ready(fwd_and_bwd_ref(all_idxs, x, w1, w2, w3))
  #for _ in range(2):
  #  jax.block_until_ready(fwd_and_bwd(all_idxs, x, w1, w2, w3, splits=8))

http://localhost:52433/data/plugin/profile/trace_viewer@;run=2025_12_01_21_33_17;tag=trace_viewer@


# remaining steps

In [13]:
#opts = dict(axis_name="x", experts_num=g, ragged_all_to_all=partial(moe.sc_kernels.ra2a, multiple=1), multiple=multiple, compute_block=compute_block)
opts = dict(axis_name="x", experts_num=g, ragged_all_to_all=jax.lax.ragged_all_to_all, multiple=multiple, compute_block=compute_block)
moe1_fn = jax.jit(partial(run_moe, **opts, gathers="builtin"))
moe2_fn = jax.jit(partial(run_moe, **opts, gathers="custom_sc"))
#moe2_fn = jax.jit(partial(run_moe, **opts, gathers="custom"))

o1, vjp1_fn = jax.vjp(partial(moe1_fn, all_idxs=all_idxs), x)
o2, vjp2_fn = jax.vjp(partial(moe2_fn, all_idxs=all_idxs), x)
vjp1_fn, vjp2_fn = jax.jit(vjp1_fn), jax.jit(vjp2_fn)

In [14]:
# np.testing.assert_allclose(x, x_new)
x_ref = jnp.repeat(x, experts_per_tok, axis=0, out_sharding=P(axis_name, None))
x_ref = x_ref.reshape((x.shape[0], experts_per_tok, *x.shape[1:]))
x_ref *= all_idxs.reshape((x.shape[0], experts_per_tok, 1, 1))
x_ref = jnp.sum(x_ref, 1)
np.testing.assert_allclose(o1, o2)
np.testing.assert_allclose(x_ref, o1)

In [15]:
r = jax.jit(lambda: jax.random.normal(jax.random.key(1), o1.shape, dtype=x.dtype),
            out_shardings=P(axis_name, None, None))()
(do1,) = vjp1_fn(r)
(do2,) = vjp2_fn(r)

In [18]:
do1_error = jnp.max(jnp.linalg.norm(do1 - do2, axis=-1) / jnp.maximum(jnp.linalg.norm(do1, axis=-1), 1e-7))
assert do1_error < 5e-3

In [ ]:
with moe.utils.profile():
  for _ in range(2):
    jax.block_until_ready(moe1_fn(x, all_idxs))
  for _ in range(2):
    jax.block_until_ready(moe2_fn(x, all_idxs))
  for _ in range(2):
    jax.block_until_ready(vjp1_fn(r))
  for _ in range(2):
    jax.block_until_ready(vjp2_fn(r))

# remaining tests

In [ ]:
@parameterized.product(experts_per_tok=[1, 2, 4, 8], device=["cpu", "tpu"])
def test_simple_moe(self, experts_per_tok, device):
  try:
    devices = jax.devices(device)
  except RuntimeError:
    self.skipTest(f"Device {device} not available")
  axis_name = "x"
  mesh = jax.make_mesh((len(devices),), (axis_name,), axis_types=jax.sharding.AxisType.Explicit, devices=devices)

  with jax.sharding.set_mesh(mesh):
    n, k, g = 256, 2048, 32
    # x, ra2a_meta = generate_data(n, k, len(devices), axis_name="x")
    # del ra2a_meta
    x = jax.random.normal(jax.random.key(0), (n, k), dtype="bfloat16")
    all_idxs = jax.random.randint(jax.random.key(0), (experts_per_tok * x.shape[0],), minval=0, maxval=g)
    x, all_idxs = jax.device_put(x, P(axis_name, None)), jax.device_put(all_idxs, P(None))
    out = run_moe(x, all_idxs, axis_name="x", experts_num=g, ragged_all_to_all=ra2a_via_ag)
    self.assertEqual(out.shape, (n, experts_per_tok, x.shape[-1]))

    x_new = np.array(out[:, 0, :])
    np.testing.assert_allclose(x, x_new)

In [ ]:
@parameterized.product(experts=[32, 128], multiple=[2, 4, 8])
def test_add_indices_works_for_moe(self, experts, multiple):
  all_idxs = jax.random.randint(jax.random.key(0), 128, minval=0, maxval=experts)
  idx_count = jnp.bincount(all_idxs, length=experts)
  pad_indices = add_indices(jnp.arange(experts), -idx_count % multiple, max_size=multiple - 1)
  # check if the pad_indices actually added the desired number of pad indices to each group
  np.testing.assert_array_equal(jnp.bincount(pad_indices, length=experts), -idx_count % multiple)

In [ ]:
@parameterized.product(experts_per_tok=[1, 2, 4], device=["cpu", "tpu"], multiple=[1, 2, 8])
def test_identity_moe_block(self, experts_per_tok, device, multiple):
  try:
    devices = jax.devices(device)
  except RuntimeError:
    self.skipTest(f"Device {device} not available")
  axis_name = "x"
  m, k, g = 4096, 128, 32
  mesh = jax.make_mesh((len(devices),), (axis_name,), axis_types=jax.sharding.AxisType.Explicit, devices=devices)

  with jax.sharding.set_mesh(mesh):
    all_idxs = jax.random.randint(jax.random.key(0), experts_per_tok * m, minval=0, maxval=g)
    x, ra2a_meta = generate_data(m, k, device_num=len(devices), axis_name=axis_name)
    del ra2a_meta
    reduce_block = lambda x: x[:, 0, ...]
    moe_fn = jax.jit(partial(run_moe, reduce_block=reduce_block, axis_name=axis_name, experts_num=g,
                              multiple=multiple, ragged_all_to_all=ra2a_via_ag))
    out = moe_fn(x, all_idxs)
    self.assertEqual(out.shape, (m, x.shape[-1]))
    np.testing.assert_array_equal(out, x)